# 01 - Data Ingestion

## Objective

This notebook ingests the monthly BTS flight datasets and the supporting reference datasets from the Databricks Volume.

The ingestion process includes:

- Loading the project configuration
- Reading and consolidating the monthly flight CSV files
- Validating the flight dataset
- Saving the consolidated flight data as a Delta table
- Reading and validating the reference CSV files
- Saving each reference dataset as a reusable Delta lookup table

The raw flight data and reference datasets are stored separately. Dataset enrichment and permanent joins will be performed during the data-cleaning stage.

#### Load project configuration

In [1]:
import importlib.util
import os
from functools import reduce
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg
from dotenv import dotenv_values
from pyspark.sql import functions as F
from pyspark.sql import types as T

DATABRICKS_PROFILE = os.getenv(
    "DATABRICKS_CONFIG_PROFILE", "capstone-serverless"
)
api_env_candidates = [
    Path.cwd() / "api" / ".env",
    Path.cwd().parent / "api" / ".env",
]
api_env_path = next(
    (candidate for candidate in api_env_candidates if candidate.is_file()),
    None,
)
api_credentials = dotenv_values(api_env_path) if api_env_path else {}
LOCAL_DATABRICKS_HOST = str(
    api_credentials.get("DATABRICKS_SERVER_HOSTNAME", "")
).strip()
LOCAL_DATABRICKS_TOKEN = str(
    api_credentials.get("DATABRICKS_ACCESS_TOKEN", "")
).strip()
if LOCAL_DATABRICKS_HOST and not LOCAL_DATABRICKS_HOST.startswith("http"):
    LOCAL_DATABRICKS_HOST = f"https://{LOCAL_DATABRICKS_HOST}"

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession

    builder = DatabricksSession.builder.serverless()
    if LOCAL_DATABRICKS_HOST and LOCAL_DATABRICKS_TOKEN:
        builder = builder.host(LOCAL_DATABRICKS_HOST).token(
            LOCAL_DATABRICKS_TOKEN
        )
    else:
        builder = builder.profile(DATABRICKS_PROFILE)
    spark = builder.getOrCreate()

try:
    dbutils
except NameError:
    from pyspark.dbutils import DBUtils

    dbutils = DBUtils(spark)

print("Project configuration loaded successfully.")

Project configuration loaded successfully.


#### Flight data ingestion

This section reads and consolidates the monthly flight CSV files stored in the raw data directory.

#### Inspect monthly files individually before combining

Each monthly flight file is read and counted separately, and its schema is compared against the first file, before any files are combined. This follows the project guidance to process monthly files individually prior to consolidation.

In [2]:
# Identify and read each monthly flight file individually

monthly_file_infos = sorted(
    (
        file_info
        for file_info in dbutils.fs.ls(cfg.RAW_PATH)
        if file_info.name.lower().endswith(
            cfg.EXPECTED_FILE_EXTENSION.lower()
        )
    ),
    key=lambda file_info: file_info.name,
)

available_raw_files = {file_info.name for file_info in monthly_file_infos}
missing_raw_files = sorted(set(cfg.EXPECTED_RAW_FILES) - available_raw_files)
unexpected_raw_files = sorted(available_raw_files - set(cfg.EXPECTED_RAW_FILES))
if missing_raw_files or unexpected_raw_files:
    raise ValueError(
        "Raw monthly files do not match the approved 2025 source set. "
        f"Missing: {missing_raw_files}; unexpected: {unexpected_raw_files}."
    )

raw_integer_columns = set(cfg.INTEGER_COLUMNS)
raw_double_columns = (
    set(cfg.DOUBLE_COLUMNS) | set(cfg.BINARY_INDICATOR_COLUMNS)
)
raw_schema = T.StructType(
    [
        T.StructField(
            column_name,
            (
                T.IntegerType()
                if column_name in raw_integer_columns
                else T.DoubleType()
                if column_name in raw_double_columns
                else T.StringType()
            ),
            True,
        )
        for column_name in cfg.EXPECTED_RAW_COLUMNS
    ]
)

monthly_dataframes = {}
monthly_summaries = []

for file_info in monthly_file_infos:
    monthly_df = (
        spark.read
        .option("header", True)
        .option("enforceSchema", False)
        .option("mode", "FAILFAST")
        .schema(raw_schema)
        .csv(file_info.path)
    )

    monthly_dataframes[file_info.name] = monthly_df
    monthly_summaries.append(
        (
            file_info.name,
            monthly_df.count(),
            len(monthly_df.columns),
            monthly_df.schema == raw_schema,
        )
    )

monthly_summary_df = spark.createDataFrame(
    monthly_summaries,
    [
        "file_name",
        "record_count",
        "column_count",
        "schema_matches_approved",
    ],
)

display(monthly_summary_df.orderBy("file_name"))


,file_name,record_count,column_count,schema_matches_approved
0,T_ONTIME_REPORTING_APRIL_2025.csv,583950,32,True
1,T_ONTIME_REPORTING_AUGUST_2025.csv,602378,32,True
2,T_ONTIME_REPORTING_DECEMBER_2025.csv,582304,32,True
3,T_ONTIME_REPORTING_FEBRUARY_2025.csv,504884,32,True
4,T_ONTIME_REPORTING_JANUARY_2025.csv,539747,32,True
5,T_ONTIME_REPORTING_JULY_2025.csv,631428,32,True
6,T_ONTIME_REPORTING_JUNE_2025.csv,611575,32,True
7,T_ONTIME_REPORTING_MARCH_2025.csv,600872,32,True
8,T_ONTIME_REPORTING_MAY_2025.csv,605648,32,True
9,T_ONTIME_REPORTING_NOVEMBER_2025.csv,570550,32,True


In [3]:
# Stop ingestion if any monthly file differs from the approved schema

files_with_schema_mismatch = (
    monthly_summary_df
    .filter(F.col("schema_matches_approved") == False)
    .count()
)

if files_with_schema_mismatch > 0:
    raise ValueError(
        "One or more monthly flight files differ from the approved "
        "32-column schema."
    )

print("All monthly flight files match the explicit approved schema.")


All monthly flight files match the explicit approved schema.


In [4]:
# Combine the individually validated monthly files into a single flight dataset

df_raw = reduce(
    lambda left, right: left.unionByName(right),
    monthly_dataframes.values(),
)

print(
    f"Flight data combined successfully from "
    f"{len(monthly_dataframes)} monthly files."
)

Flight data combined successfully from 12 monthly files.


#### Validate flight dataset

In [5]:
# Validate that the consolidated flight dataset is not empty

flight_record_count = df_raw.count()
flight_column_count = len(df_raw.columns)

if flight_record_count == 0:
    raise RuntimeError(
        "The consolidated flight dataset is empty."
    )

if flight_column_count == 0:
    raise RuntimeError(
        "The consolidated flight dataset contains no columns."
    )

print(f"Total flight records: {flight_record_count:,}")
print(f"Total flight columns: {flight_column_count}")

Total flight records: 7,001,619
Total flight columns: 32


In [6]:
# Validate that the combined dataset preserves every monthly record

expected_total_records = sum(
    summary[1] for summary in monthly_summaries
)

if flight_record_count != expected_total_records:
    raise RuntimeError(
        "The combined flight dataset record count does not match the "
        f"sum of the individual monthly files "
        f"({flight_record_count:,} vs {expected_total_records:,})."
    )

print(
    "Combined record count reconciled against monthly file totals: "
    f"{flight_record_count:,}"
)

Combined record count reconciled against monthly file totals: 7,001,619


#### Preview flight dataset

In [7]:
display(df_raw.limit(10))

,QUARTER,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2,4,1,4/7/2025 12:00:00 AM,AA,1,JFK,"New York, NY",New York,LAX,"Los Angeles, CA",California,700,-6.0,0.0,23.0,6.0,1020,-34.0,0.0,0.0,None,0.0,380.0,352.0,323.0,2475.0,NaN,NaN,NaN,NaN,NaN
1,2,4,1,4/7/2025 12:00:00 AM,AA,10,LAX,"Los Angeles, CA",California,JFK,"New York, NY",New York,2200,-11.0,0.0,15.0,8.0,626,-24.0,0.0,0.0,None,0.0,326.0,313.0,290.0,2475.0,NaN,NaN,NaN,NaN,NaN
2,2,4,1,4/7/2025 12:00:00 AM,AA,1002,MSN,"Madison, WI",Wisconsin,CLT,"Charlotte, NC",North Carolina,620,7.0,0.0,13.0,14.0,935,8.0,0.0,0.0,None,0.0,135.0,136.0,109.0,708.0,NaN,NaN,NaN,NaN,NaN
3,2,4,1,4/7/2025 12:00:00 AM,AA,1003,CLT,"Charlotte, NC",North Carolina,MCI,"Kansas City, MO",Missouri,1615,2.0,0.0,22.0,11.0,1745,3.0,0.0,0.0,None,0.0,150.0,151.0,118.0,808.0,NaN,NaN,NaN,NaN,NaN
4,2,4,1,4/7/2025 12:00:00 AM,AA,1003,MCI,"Kansas City, MO",Missouri,CLT,"Charlotte, NC",North Carolina,1837,-5.0,0.0,16.0,20.0,2154,-5.0,0.0,0.0,None,0.0,137.0,137.0,101.0,808.0,NaN,NaN,NaN,NaN,NaN
5,2,4,1,4/7/2025 12:00:00 AM,AA,1007,CLT,"Charlotte, NC",North Carolina,STL,"St. Louis, MO",Missouri,2029,-5.0,0.0,15.0,5.0,2124,-13.0,0.0,0.0,None,0.0,115.0,107.0,87.0,575.0,NaN,NaN,NaN,NaN,NaN
6,2,4,1,4/7/2025 12:00:00 AM,AA,1009,LAX,"Los Angeles, CA",California,ORD,"Chicago, IL",Illinois,2305,-4.0,0.0,13.0,11.0,514,-11.0,0.0,0.0,None,0.0,249.0,242.0,218.0,1744.0,NaN,NaN,NaN,NaN,NaN
7,2,4,1,4/7/2025 12:00:00 AM,AA,1010,DFW,"Dallas/Fort Worth, TX",Texas,STL,"St. Louis, MO",Missouri,2040,37.0,1.0,19.0,5.0,2228,32.0,1.0,0.0,None,0.0,108.0,103.0,79.0,550.0,22.0,0.0,0.0,0.0,10.0
8,2,4,1,4/7/2025 12:00:00 AM,AA,1011,DFW,"Dallas/Fort Worth, TX",Texas,LGA,"New York, NY",New York,1742,25.0,1.0,18.0,6.0,2200,18.0,1.0,0.0,None,0.0,198.0,191.0,167.0,1389.0,0.0,0.0,0.0,0.0,18.0
9,2,4,1,4/7/2025 12:00:00 AM,AA,1012,CLT,"Charlotte, NC",North Carolina,LAX,"Los Angeles, CA",California,1654,86.0,1.0,15.0,7.0,1915,52.0,1.0,0.0,None,0.0,321.0,287.0,265.0,2125.0,16.0,0.0,0.0,0.0,36.0


#### Review flight schema

In [8]:
# Review the schema inferred from the monthly CSV files

df_raw.printSchema()

root
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_NM: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DEL15: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELA

#### Save raw flight table

In [9]:
# Store the consolidated flight dataset as a Delta table

(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.RAW_TABLE)
)

print(f"Raw flight table created successfully: {cfg.RAW_TABLE}")

Raw flight table created successfully: workspace.default.flights_raw


#### Validate saved flight table

In [10]:
# Confirm that the saved Delta table contains the expected records

saved_flight_count = spark.read.table(
    cfg.RAW_TABLE
).count()

if saved_flight_count != flight_record_count:
    raise RuntimeError(
        "The saved flight-table record count does not match "
        "the ingested dataset."
    )

print(
    f"Saved flight-table records confirmed: "
    f"{saved_flight_count:,}"
)

Saved flight-table records confirmed: 7,001,619


## Reference data ingestion

This section reads the supporting reference CSV files and stores each dataset as a separate Delta lookup table.

The reference datasets provide descriptive values for coded flight attributes, including airlines, airports, cancellation reasons, months, quarters, weekdays, and binary indicators.

#### Load and inspect reference datasets

In [11]:
# Load each configured reference CSV file

reference_dataframes = {}
reference_summaries = []

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_path = reference_config["path"]

    reference_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(reference_path)
    )

    record_count = reference_df.count()
    column_count = len(reference_df.columns)

    if record_count == 0:
        raise RuntimeError(
            f"Reference dataset '{reference_name}' is empty."
        )

    if column_count == 0:
        raise RuntimeError(
            f"Reference dataset '{reference_name}' "
            f"contains no columns."
        )

    reference_dataframes[reference_name] = reference_df

    reference_summaries.append(
        (
            reference_name,
            reference_path,
            record_count,
            column_count,
            ", ".join(reference_df.columns),
        )
    )

print(
    f"Reference datasets loaded successfully: "
    f"{len(reference_dataframes)}"
)

Reference datasets loaded successfully: 7


#### Review reference dataset summary

In [12]:
# Create a summary of the loaded reference datasets

reference_summary_df = spark.createDataFrame(
    reference_summaries,
    [
        "reference_name",
        "source_path",
        "record_count",
        "column_count",
        "columns",
    ],
)

display(
    reference_summary_df.orderBy("reference_name")
)

,reference_name,source_path,record_count,column_count,columns
0,airlines,/Volumes/workspace/default/flight_delay_capstone/reference/L_UNIQUE_CARRIERS_Reporting_Airline.csv,1776,2,"Code, Description"
1,airports,/Volumes/workspace/default/flight_delay_capstone/reference/L_AIRPORT_Origin_Dest.csv,6914,2,"Code, Description"
2,cancellation_codes,/Volumes/workspace/default/flight_delay_capstone/reference/L_CANCELLATION_CancellationCode.csv,4,2,"Code, Description"
3,months,/Volumes/workspace/default/flight_delay_capstone/reference/L_MONTHS_Month.csv,12,2,"Code, Description"
4,quarters,/Volumes/workspace/default/flight_delay_capstone/reference/L_QUARTERS_Quarter.csv,4,2,"Code, Description"
5,weekdays,/Volumes/workspace/default/flight_delay_capstone/reference/L_WEEKDAYS_DayOfWeek.csv,8,2,"Code, Description"
6,yes_no,/Volumes/workspace/default/flight_delay_capstone/reference/L_YESNO_RESP_ArrDel15_DepDel15_Cancelled_Diverted.csv,2,2,"Code, Description"


#### Validate reference schemas

In [13]:
# Confirm that each reference dataset contains Code and Description

required_reference_columns = {
    "Code",
    "Description",
}

for reference_name, reference_df in (
    reference_dataframes.items()
):
    available_columns = set(reference_df.columns)

    missing_columns = (
        required_reference_columns
        - available_columns
    )

    if missing_columns:
        raise ValueError(
            f"Reference dataset '{reference_name}' is missing "
            f"the following required columns: "
            f"{sorted(missing_columns)}"
        )

print(
    "All reference datasets contain the required "
    "'Code' and 'Description' columns."
)

All reference datasets contain the required 'Code' and 'Description' columns.


#### Validate reference codes

In [14]:
# Check for null and duplicate codes in each reference dataset

reference_quality_results = []

for reference_name, reference_df in (
    reference_dataframes.items()
):
    total_records = reference_df.count()

    null_code_count = (
        reference_df
        .filter(F.col("Code").isNull())
        .count()
    )

    distinct_code_count = (
        reference_df
        .select("Code")
        .distinct()
        .count()
    )

    duplicate_code_count = (
        total_records
        - distinct_code_count
    )

    reference_quality_results.append(
        (
            reference_name,
            total_records,
            null_code_count,
            duplicate_code_count,
        )
    )

reference_quality_df = spark.createDataFrame(
    reference_quality_results,
    [
        "reference_name",
        "record_count",
        "null_code_count",
        "duplicate_code_count",
    ],
)

display(
    reference_quality_df.orderBy("reference_name")
)

,reference_name,record_count,null_code_count,duplicate_code_count
0,airlines,1776,0,0
1,airports,6914,0,0
2,cancellation_codes,4,0,0
3,months,12,0,0
4,quarters,4,0,0
5,weekdays,8,0,0
6,yes_no,2,0,0


In [15]:
# Stop ingestion if reference keys contain quality issues

invalid_reference_datasets = (
    reference_quality_df
    .filter(
        (F.col("null_code_count") > 0)
        | (F.col("duplicate_code_count") > 0)
    )
    .count()
)

if invalid_reference_datasets > 0:
    raise ValueError(
        "One or more reference datasets contain null or "
        "duplicate codes. Review the quality results before "
        "saving the lookup tables."
    )

print("Reference-code quality validation completed successfully.")

Reference-code quality validation completed successfully.


#### Preview reference datasets

In [16]:
# Display a small sample from each reference dataset

for reference_name, reference_df in (
    reference_dataframes.items()
):
    print(f"Reference dataset: {reference_name}")
    display(reference_df.limit(10))

Reference dataset: airports


,Code,Description
0,01A,"Afognak Lake, AK: Afognak Lake Airport"
1,03A,"Granite Mountain, AK: Bear Creek Mining Strip"
2,04A,"Lik, AK: Lik Mining Camp"
3,05A,"Little Squaw, AK: Little Squaw Airport"
4,05K,"Port Alsworth, AK: Wilder Runway"
5,06A,"Kizhuyak, AK: Kizhuyak Bay"
6,07A,"Klawock, AK: Klawock Seaplane Base"
7,08A,"Elizabeth Island, AK: Elizabeth Island Airport"
8,09A,"Homer, AK: Augustin Island"
9,1AK,"Mertarvik, AK: Mertarvik Quarry Road Landing Strip"


Reference dataset: cancellation_codes


,Code,Description
0,A,Carrier
1,B,Weather
2,C,National Air System
3,D,Security


Reference dataset: months


...

,Code,Description
0,02Q,Titan Airways
1,05Q,"Comlux Aviation, AG"
2,06Q,Master Top Linhas Aereas Ltd.
3,07Q,Flair Airlines Ltd.
4,09Q,"Swift Air, LLC d/b/a Eastern Air Lines d/b/a Eastern"
5,0BQ,DCA
6,0CQ,ACM AIR CHARTER GmbH
7,0FQ,"Maine Aviation Aircraft Charter, LLC"
8,0GQ,"Inter Island Airways, d/b/a Inter Island Air"
9,0HQ,Polar Airlines de Mexico d/b/a Nova Air


Reference dataset: weekdays


,Code,Description
0,1,Monday
1,2,Tuesday
2,3,Wednesday
3,4,Thursday
4,5,Friday
5,6,Saturday
6,7,Sunday
7,9,Unknown


Reference dataset: yes_no


,Code,Description
0,0,No
1,1,Yes


#### Save reference lookup tables

In [17]:
# Store each reference dataset as a Delta lookup table

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_table = reference_config["table"]
    reference_df = reference_dataframes[reference_name]

    (
        reference_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(reference_table)
    )

    print(
        f"Lookup table created successfully: "
        f"{reference_table}"
    )

Lookup table created successfully: workspace.default.airports_lookup
Lookup table created successfully: workspace.default.cancellation_codes_lookup
Lookup table created successfully: workspace.default.months_lookup
Lookup table created successfully: workspace.default.quarters_lookup
Lookup table created successfully: workspace.default.airlines_lookup
Lookup table created successfully: workspace.default.weekdays_lookup
Lookup table created successfully: workspace.default.yes_no_lookup


#### Validate saved lookup tables

In [18]:
# Confirm that each saved lookup table preserves its record count

lookup_validation_results = []

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_table = reference_config["table"]

    source_count = reference_dataframes[
        reference_name
    ].count()

    saved_count = spark.read.table(
        reference_table
    ).count()

    counts_match = source_count == saved_count

    lookup_validation_results.append(
        (
            reference_name,
            reference_table,
            source_count,
            saved_count,
            counts_match,
        )
    )

lookup_validation_df = spark.createDataFrame(
    lookup_validation_results,
    [
        "reference_name",
        "table_name",
        "source_record_count",
        "saved_record_count",
        "counts_match",
    ],
)

display(
    lookup_validation_df.orderBy("reference_name")
)

,reference_name,table_name,source_record_count,saved_record_count,counts_match
0,airlines,workspace.default.airlines_lookup,1776,1776,True
1,airports,workspace.default.airports_lookup,6914,6914,True
2,cancellation_codes,workspace.default.cancellation_codes_lookup,4,4,True
3,months,workspace.default.months_lookup,12,12,True
4,quarters,workspace.default.quarters_lookup,4,4,True
5,weekdays,workspace.default.weekdays_lookup,8,8,True
6,yes_no,workspace.default.yes_no_lookup,2,2,True


In [19]:
# Stop execution if any lookup-table count does not match

failed_lookup_validations = (
    lookup_validation_df
    .filter(F.col("counts_match") == False)
    .count()
)

if failed_lookup_validations > 0:
    raise RuntimeError(
        "One or more lookup-table record counts do not "
        "match their source datasets."
    )

print("All lookup-table record counts were validated successfully.")

All lookup-table record counts were validated successfully.


#### Data ingestion completion

In [20]:
print("Data ingestion completed successfully.")
print(f"Raw flight table: {cfg.RAW_TABLE}")
print(
    f"Reference lookup tables created: "
    f"{len(cfg.REFERENCE_DATASETS)}"
)

Data ingestion completed successfully.
Raw flight table: workspace.default.flights_raw
Reference lookup tables created: 7
